<a href="https://colab.research.google.com/github/gouthamgo/AI-projects/blob/main/Finetune_Gemma_Embedding_300M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U sentence-transformers
!pip install git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-bm73tfn7
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-bm73tfn7
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from huggingface_hub import login
login()


In [3]:
triplets = [
  {
    "anchor": "How do I join and check my rewards in the Rewards Program?",
    "positive": "Sign in to your account to view your rewards value and expiry. Earn points for every $1 spent on full-priced items. There are four tiers of benefits.",
    "negative": "Gift cards are valid for 36 months and can't be replaced if lost or stolen."
  },
  {
    "anchor": "What are the delivery costs for Australian orders?",
    "positive": "Australian Standard Delivery is FREE for orders over $90, or $9.95 flat rate for orders under $90. Express Delivery is $14.95 flat rate.",
    "negative": "Afterpay can be used for purchases up to $4,000 and combined with Rewards Vouchers."
  },
  {
    "anchor": "How do I track my order delivery?",
    "positive": "You can check your order’s status using the link sent in your dispatch email or via the AusPost app. Track using the email registered to your order.",
    "negative": "Returns must follow the Returns Policy and be within 30 days of dispatch date."
  },
  {
    "anchor": "How does Click & Collect work? What is the bonus voucher?",
    "positive": "Click & Collect is free. Pickup is within 5–11 business days at your allocated store. Pick up your parcel to receive a $10 voucher per item (minimum $10 spend). Voucher valid in stores only.",
    "negative": "PayPal is available for purchases in stores and online up to $2,000."
  },
  {
    "anchor": "What is the returns policy for online purchases?",
    "positive": "Online purchases can be returned within 30 days of the order dispatch if eligible under our policy. Voucher returns are issued as credit notes.",
    "negative": "Apple Pay can be used in stores and online with Mastercard, Visa, or Amex cards."
  },
  {
    "anchor": "What payment options does Taking Shape accept?",
    "positive": "American Express, Visa, MasterCard, Taking Shape Gift Cards, PayPal, PayPal Pay in 4, Apple Pay, and Afterpay are accepted.",
    "negative": "Rewards points are earned on full-priced items only; membership tiers grant increased benefits."
  },
  {
    "anchor": "What are the gift card rules and validity?",
    "positive": "Gift Cards can only be redeemed in stores & online in the currency of issue. Not valid in Myer stores. They’re valid for 36 months, non‑refundable, and can't be replaced if lost or stolen.",
    "negative": "International shipping uses Global-e; select your shipping country with the flag icon."
  },
  {
    "anchor": "How can I make my online shopping more secure?",
    "positive": "Verify the official website, use strong passwords, beware of phishing, check accounts regularly, use secure Wi-Fi and enable two-factor authentication.",
    "negative": "Credit notes are valid for 12 months and only redeemable in-store."
  },
  {
    "anchor": "How do I change my account password?",
    "positive": "Login, select 'My Details', choose CHANGE MY PASSWORD, update your password and save. Use a strong password with letters, numbers, and symbols.",
    "negative": "Online orders can’t be amended or cancelled once placed."
  },
  {
    "anchor": "Does Taking Shape ship internationally?",
    "positive": "Yes, we use Global-e for international shipping. Choose your country via the flag icon for local payment and rates.",
    "negative": "Click & Collect is not available for delivery to Myer or online-only stores."
  }
]


In [4]:
from datasets import Dataset
train_dataset = Dataset.from_list(triplets)


In [7]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "google/embeddinggemma-300M"
model = SentenceTransformer(model_id).to(device=device)


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

print("--- BEFORE FINETUNE ---")
for item in triplets:
    q = item["anchor"]
    pos = item["positive"]
    neg = item["negative"]
    emb_q = model.encode([q])
    emb_pos = model.encode([pos])
    emb_neg = model.encode([neg])
    score_pos = cosine_similarity(emb_q, emb_pos)[0][0]
    score_neg = cosine_similarity(emb_q, emb_neg)[0][0]
    print(f"Q: {q}\nPositive Score: {score_pos:.3f}\nNegative Score: {score_neg:.3f}\n")


--- BEFORE FINETUNE ---
Q: How do I join and check my rewards in the Rewards Program?
Positive Score: 0.568
Negative Score: 0.207

Q: What are the delivery costs for Australian orders?
Positive Score: 0.741
Negative Score: 0.323

Q: How do I track my order delivery?
Positive Score: 0.703
Negative Score: 0.411

Q: How does Click & Collect work? What is the bonus voucher?
Positive Score: 0.621
Negative Score: 0.197

Q: What is the returns policy for online purchases?
Positive Score: 0.763
Negative Score: 0.195

Q: What payment options does Taking Shape accept?
Positive Score: 0.697
Negative Score: 0.245

Q: What are the gift card rules and validity?
Positive Score: 0.593
Negative Score: 0.195

Q: How can I make my online shopping more secure?
Positive Score: 0.601
Negative Score: 0.207

Q: How do I change my account password?
Positive Score: 0.732
Negative Score: 0.311

Q: Does Taking Shape ship internationally?
Positive Score: 0.544
Negative Score: 0.295



In [9]:
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments

loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir="./finetuned-faq-model",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    report_to="none",
    save_strategy="epoch",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)
trainer.train()


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


TrainOutput(global_step=8, training_loss=0.04563778266310692, metrics={'train_runtime': 284.3439, 'train_samples_per_second': 0.141, 'train_steps_per_second': 0.028, 'total_flos': 0.0, 'train_loss': 0.04563778266310692, 'epoch': 4.0})

In [14]:
# Save the finetuned model
output_path = "./finetuned-faq-model-saved"
model.save(output_path)
print(f"Finetuned model saved to {output_path}")

Finetuned model saved to ./finetuned-faq-model-saved


In [24]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch

# Load the original model for comparison
device = "cuda" if torch.cuda.is_available() else "cpu"
model_original = SentenceTransformer("google/embeddinggemma-300M").to(device=device)


print("--- BEFORE vs AFTER FINETUNE ---")
for item in triplets:
    q = item["anchor"]
    pos = item["positive"]
    neg = item["negative"]

    emb_q_original = model_original.encode([q])
    emb_pos_original = model_original.encode([pos])
    emb_neg_original = model_original.encode([neg])
    score_pos_original = cosine_similarity(emb_q_original, emb_pos_original)[0][0]
    score_neg_original = cosine_similarity(emb_q_original, emb_neg_original)[0][0]

    emb_q_finetuned = model.encode([q])
    emb_pos_finetuned = model.encode([pos])
    emb_neg_finetuned = model.encode([neg])
    score_pos_finetuned = cosine_similarity(emb_q_finetuned, emb_pos_finetuned)[0][0]
    score_neg_finetuned = cosine_similarity(emb_q_finetuned, emb_neg_finetuned)[0][0]

    print(f"Q: {q}")
    print(f"  Original: Positive Score: {score_pos_original:.3f}, Negative Score: {score_neg_original:.3f}")
    print(f"  Finetuned: Positive Score: {score_pos_finetuned:.3f}, Negative Score: {score_neg_finetuned:.3f}\n")

--- BEFORE vs AFTER FINETUNE ---
Q: How do I join and check my rewards in the Rewards Program?
  Original: Positive Score: 0.568, Negative Score: 0.207
  Finetuned: Positive Score: 0.754, Negative Score: -0.124

Q: What are the delivery costs for Australian orders?
  Original: Positive Score: 0.741, Negative Score: 0.323
  Finetuned: Positive Score: 0.871, Negative Score: 0.097

Q: How do I track my order delivery?
  Original: Positive Score: 0.703, Negative Score: 0.411
  Finetuned: Positive Score: 0.819, Negative Score: 0.155

Q: How does Click & Collect work? What is the bonus voucher?
  Original: Positive Score: 0.621, Negative Score: 0.197
  Finetuned: Positive Score: 0.912, Negative Score: -0.088

Q: What is the returns policy for online purchases?
  Original: Positive Score: 0.763, Negative Score: 0.195
  Finetuned: Positive Score: 0.901, Negative Score: -0.023

Q: What payment options does Taking Shape accept?
  Original: Positive Score: 0.697, Negative Score: 0.245
  Finetuned